In [0]:
%run "../Modulo 2/Configurando o storage"

In [0]:
caminho_bronze = "abfss://bronze@datalakexperiun.dfs.core.windows.net/"
caminho_silver = "abfss://silver@datalakexperiun.dfs.core.windows.net/"
caminho_gold = "abfss://gold@datalakexperiun.dfs.core.windows.net/"
caminho_arquivo_jogadores = f"{caminho_bronze}jogadores.json"


In [0]:
from pyspark.sql.types import StructField, StructType, StringType, DoubleType, BooleanType, IntegerType

In [0]:
schema_jogadores = StructType(
    fields=[
        StructField("id", IntegerType(), True),
        StructField("name", StringType(), True),
        StructField("firstname", StringType(), True),
        StructField("lastname", StringType(), True),

        StructField("age", IntegerType(), True),

        StructField(
            "birth",
            StructType([
                StructField("date", StringType(), True),
                StructField("place", StringType(), True),
                StructField("country", StringType(), True)
            ]),
            True
        ),

        StructField("nationality", StringType(), True),
        StructField("height", StringType(), True),
        StructField("weight", StringType(), True),
        StructField("injured", BooleanType(), True),
        StructField("photo", StringType(), True)
    ]
)

In [0]:
jogadores_df = spark.read.schema(schema_jogadores).json(caminho_arquivo_jogadores)

In [0]:
display(jogadores_df)

In [0]:
jogadores_df.printSchema()

In [0]:
jogadores_df_renomeado = jogadores_df \
    .withColumnRenamed("id", "id_jogador") \
    .withColumnRenamed("name", "nm_jogador") \
    .withColumnRenamed("firstname", "nm_primeiro") \
    .withColumnRenamed("lastname", "nm_sobrenome") \
    .withColumnRenamed("age", "idade") \
    .withColumnRenamed("birth", "dados_nascimento") \
    .withColumnRenamed("nationality", "nm_nacionalidade") \
    .withColumnRenamed("height", "altura") \
    .withColumnRenamed("weight", "peso") \
    .withColumnRenamed("injured", "e_lesionado") \
    .withColumnRenamed("photo", "link_foto")

In [0]:
from pyspark.sql.functions import current_timestamp

In [0]:
jogadores_df_data = jogadores_df_renomeado.withColumn('dt_ingestao', current_timestamp())

In [0]:
display(jogadores_df_data)

In [0]:
jogadores_df_data.write.parquet(f"{caminho_silver}jogadores/", mode="overwrite")